In [1]:
import re
import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords

from keras.models import Model
from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences, to_categorical
from keras.layers import Dense, LSTM, Dropout, Input, Embedding

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [2]:
MAX_SEQUENCE_LENGTH = 50
MAX_NB_WORDS = 2000000
EMBEDDING_DIM = 100
VALIDATION_SPLIT = 0.2

In [3]:
df = pd.read_csv('Emotion_data.csv')

In [4]:
data = df.sample(n=20000)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 20000 entries, 465 to 21308
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Text     20000 non-null  object
 1   Emotion  20000 non-null  object
dtypes: object(2)
memory usage: 468.8+ KB


In [6]:
data.head()

,Text,Emotion
465,im not only thankful that everything seems to ...,happy
5228,i feel little comes from my divine center,happy
10297,i got a feeling like something tragic is going...,sadness
16582,im happy but i feel all this pressure to do on...,sadness
5139,i feel no matter how convinced i am that i am ...,happy


In [7]:
wl = nltk.WordNetLemmatizer()
ps = nltk.PorterStemmer()

In [8]:
def preprocess(text):
    stop = stopwords.words('english')
    text = text.lower()
    text = re.sub('[^a-zA-Z\s]','',text)
    text = [word for word in text.split() if word not in stop]
    text = [wl.lemmatize(word) for word in text]
    text = [ps.stem(word) for word in text]
    return text
    

In [9]:
type(data['Text'])

pandas.core.series.Series

In [10]:
processed_data = data['Text'].map(preprocess)

In [11]:
processed_data.head()

465      [im, thank, everyth, seem, work, wrap, week, n...
5228                    [feel, littl, come, divin, center]
10297    [got, feel, like, someth, tragic, go, happen, ...
16582    [im, happi, feel, pressur, one, thing, anoth, ...
5139     [feel, matter, convinc, alon, life, journey, m...
Name: Text, dtype: object

In [12]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(processed_data)
sequences= tokenizer.texts_to_sequences(processed_data)

word_index = tokenizer.word_index
len(word_index)

12694

In [13]:
sequences[:5]

[[3, 159, 86, 97, 30, 1728, 67, 87, 250, 43, 1, 38, 420, 14, 21],
 [1, 12, 39, 539, 1428],
 [59,
  1,
  2,
  29,
  807,
  9,
  121,
  3,
  843,
  158,
  3,
  2,
  4757,
  3,
  137,
  201,
  16,
  86,
  316],
 [3, 54, 1, 267, 16, 17, 176, 308, 8, 382],
 [1, 312, 362, 138, 21, 1201, 556, 138]]

In [14]:
max([len(i) for i in sequences])

35

In [15]:
data['Emotion'].value_counts()

happy       6546
sadness     5831
anger       2783
fear        2484
love        1542
surprise     814
Name: Emotion, dtype: int64

In [16]:
features = pad_sequences(sequences, MAX_SEQUENCE_LENGTH)
labels = pd.get_dummies(data['Emotion'])
features.shape, labels.shape

((20000, 50), (20000, 6))

In [17]:
labels.head()

,anger,fear,happy,love,sadness,surprise
465,0,0,1,0,0,0
5228,0,0,1,0,0,0
10297,0,0,0,0,1,0
16582,0,0,0,0,1,0
5139,0,0,1,0,0,0


In [18]:
x_train, x_test, y_train, y_test = train_test_split(features, labels, test_size = 0.2, random_state = 20)
x_test, x_val, y_test, y_val = train_test_split( features, labels, test_size=0.50, random_state=4)
x_train.shape, x_test.shape, x_val.shape, y_train.shape, y_test.shape, y_val.shape

((16000, 50), (10000, 50), (10000, 50), (16000, 6), (10000, 6), (10000, 6))

In [ ]:
embedding_layer = Embedding(len(word_index)+1,Embedding_dim,
                            input_length=Max_sequence_length)(input_sequences)

In [19]:
convs = []
filter_sizes = [3,4,5]

sequence_input = Input(shape=(MAX_SEQUENCE_LENGTH,))
embedded_sequences = embedding_layer(sequence_input)

for fsz in filter_sizes:
    l_conv = Conv1D(128,fsz,activation='relu')(embedded_sequences)
    l_pool = MaxPooling1D(5)(l_conv)
    convs.append(l_pool)   
l_merge = Concatenate()(convs)
l_cov1= Conv1D(filters=128, kernel_size=5, activation='relu')(l_merge)
l_pool1 = MaxPooling1D(5)(l_cov1)
# l_cov2 = Conv1D(filters=128, kernel_size=5, activation='relu')(l_pool1)
# l_pool2 = MaxPooling1D(30)(l_cov2)
l_flat = Flatten()(l_pool1)
l_dense = Dense(128, activation='relu')(l_flat)
preds = Dense(13, activation='softmax')(l_dense)

model = Model(sequence_input, preds)
model.compile(loss='binary_crossentropy',
              optimizer='Nadam',
              metrics=['acc'])

model.summary()


In [22]:
history = model.fit(x_train, y_train, validation_data=(x_val,y_val), epochs=25, batch_size=150, verbose=1)

Epoch 1/25
107/107 [==============================] - 215s 1s/step - loss: 1.5628 - acc: 0.3644 - val_loss: 1.2145 - val_acc: 0.5393
Epoch 2/25
107/107 [==============================] - 113s 1s/step - loss: 0.7081 - acc: 0.7609 - val_loss: 0.3208 - val_acc: 0.8944
Epoch 3/25
107/107 [==============================] - 90s 841ms/step - loss: 0.2547 - acc: 0.9170 - val_loss: 0.2039 - val_acc: 0.9324
Epoch 4/25
107/107 [==============================] - 77s 719ms/step - loss: 0.1510 - acc: 0.9500 - val_loss: 0.1454 - val_acc: 0.9526
Epoch 5/25
107/107 [==============================] - 75s 704ms/step - loss: 0.0994 - acc: 0.9655 - val_loss: 0.1635 - val_acc: 0.9462
Epoch 6/25
107/107 [==============================] - 74s 694ms/step - loss: 0.0764 - acc: 0.9747 - val_loss: 0.1398 - val_acc: 0.9577
Epoch 7/25
107/107 [==============================] - 74s 697ms/step - loss: 0.0603 - acc: 0.9799 - val_loss: 0.1299 - val_acc: 0.9616
Epoch 8/25
107/107 [==============================] - 74s 6

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline 
# list all data in history
print(history.history.keys())
# summarize history for accuracy
plt.plot(history.history['acc'])
plt.plot(history.history['val_acc'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [23]:
y_pred = model.predict(x_test, verbose=1)

313/313 [==============================] - 37s 105ms/step


In [25]:
def fx(array):
    return np.argmax(array)

temp = np.array(y_test)
st_pred = y_pred.tolist()
st_temp = temp.tolist()

arr = np.array(list(map(fx,st_temp)))
ans = np.array(list(map(fx,st_pred)))

In [26]:
f1 = f1_score(arr,ans, average=None)
f2 = f1_score(arr,ans, average='macro')
f3 =  f1_score(arr,ans, average='micro')
f4 = f1_score(arr,ans, average='weighted')
f5 = accuracy_score(arr,ans)

In [27]:
print(f1)
print(f2)
print(f3)
print(f4)
print('acc',f5)

[0.96720167 0.9618933  0.97544402 0.92866242 0.98169137 0.93844367]
0.9588894084694813
0.9691
0.9691796965912618
acc 0.9691


In [28]:
p1 = precision_score(arr,ans, average='weighted')
r1 = recall_score(arr,ans,average='weighted')
p1,r1

(0.9694142959453146, 0.9691)